In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import cv2
from skimage import morphology
from skimage.filters import threshold_otsu

import geopandas as gpd
from rasterio.mask import mask


# ===================================================
# CONFIG
# ===================================================
raster_path = r"C:\Users\LENOVO\Downloads\S2_2024-11-11.tif"
poly_path   = r"C:\Users\LENOVO\Downloads\RecorteLaguna.geojson"  # o .shp

MIN_OBJ_SIZE = 150
HOLE_AREA    = 200
CLOSING_DISK = 2

# Si el agua aparece más OSCURA en GRAY_clahe -> True
# Si el agua aparece más CLARA -> False (cambia a False si sale invertido)
WATER_IS_DARK = True

# ===================================================
# 1) ABRIR RASTER y LEER POLÍGONO
# ===================================================
with rasterio.open(raster_path) as src:
    print("CRS raster:", src.crs)

    gdf = gpd.read_file(poly_path)
    if gdf.empty:
        raise ValueError("El archivo del polígono está vacío.")

    if gdf.crs is None:
        raise ValueError("Tu polígono no tiene CRS definido. En QGIS asigna CRS y vuelve a exportar.")

    # Unir por si hay varios polígonos
    geom = gdf.geometry.unary_union

    # Reproyectar polígono al CRS del raster (CLAVE)
    geom = gpd.GeoSeries([geom], crs=gdf.crs).to_crs(src.crs).iloc[0]
    geoms = [geom.__geo_interface__]

    # Recorte por polígono -> devuelve MaskedArray (fuera del polígono queda en máscara)
    out_img, out_transform = mask(src, geoms, crop=True, filled=False)

# ROI: True dentro del polígono
roi = ~out_img[0].mask

# ===================================================
# 2) LEER BANDAS (fuera del polígono -> NaN)
# ===================================================
# out_img shape: (bandas, filas, cols)
B2  = np.ma.filled(out_img[0].astype("float32"), np.nan)  # Azul
B3  = np.ma.filled(out_img[1].astype("float32"), np.nan)  # Verde
B4  = np.ma.filled(out_img[2].astype("float32"), np.nan)  # Rojo
B8  = np.ma.filled(out_img[3].astype("float32"), np.nan)  # NIR
B11 = np.ma.filled(out_img[4].astype("float32"), np.nan)  # SWIR1

plt.figure(figsize=(6,6))
plt.imshow(B3, cmap="Greens")
plt.title("Recorte (bounding box) con ROI del polígono (B3)")
plt.colorbar()
plt.show()

# ===================================================
# 3) IMAGEN SINTÉTICA EN GRISES
# ===================================================
GRAY = (0.30 * B2 +
        0.30 * B3 +
        0.20 * B4 -
        0.10 * B8 -
        0.10 * B11)

plt.figure(figsize=(6,6))
plt.imshow(GRAY, cmap="gray")
plt.title("GRAY (NaN fuera del polígono)")
plt.colorbar()
plt.show()

# ===================================================
# 4) NORMALIZACIÓN A 8-BIT (SOLO ROI)
#    Importante: NO convertir NaN a median para Otsu global
# ===================================================
vals = GRAY[roi]
vals = vals[np.isfinite(vals)]
if vals.size == 0:
    raise ValueError("No hay valores válidos dentro del polígono. Revisa que el polígono caiga sobre el raster.")

# Normalización robusta para evitar outliers
gmin, gmax = np.percentile(vals, (2, 98))
if gmax <= gmin:
    gmin, gmax = float(np.min(vals)), float(np.max(vals))
    if gmax <= gmin:
        raise ValueError("Rango inválido dentro del ROI. Revisa datos del raster.")

GRAY_clip = np.clip(GRAY, gmin, gmax)

# Creamos 8-bit, fuera queda 0 por defecto
GRAY_8 = np.zeros_like(GRAY_clahe, dtype=np.uint8)
GRAY_8[roi] = ((GRAY_clip[roi] - gmin) / (gmax - gmin) * 255.0).astype(np.uint8)

plt.figure(figsize=(6,6))
plt.imshow(GRAY_8, cmap="gray")
plt.title("GRAY_8 (0 fuera del polígono)")
plt.colorbar()
plt.show()

# ===================================================
# 5) CLAHE (mejor contraste)
#    Para que CLAHE no se “dañe” por el 0 afuera,
#    llenamos afuera con la mediana del ROI y luego volvemos a poner 0 afuera.
# ===================================================
median8 = int(np.median(GRAY_8[roi]))
GRAY_8_for_clahe = GRAY_8.copy()
GRAY_8_for_clahe[~roi] = median8

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
GRAY_clahe = clahe.apply(GRAY_8_for_clahe)

# Para visualizar, ponemos 0 afuera (negro)
GRAY_clahe_viz = GRAY_clahe.copy()
GRAY_clahe_viz[~roi] = 255 

plt.figure(figsize=(6,6))
plt.imshow(GRAY_clahe_viz, cmap="gray")
plt.title("CLAHE ")
plt.colorbar()
plt.show()

# ===================================================
# 6) OTSU SOLO DENTRO DEL POLÍGONO (ROI)
#    -> fuera del polígono JAMÁS será agua
# ===================================================
thr = threshold_otsu(GRAY_clahe[roi])
print("Umbral Otsu (solo ROI):", thr)

otsu_mask = np.zeros_like(GRAY_clahe, dtype=np.uint8)

if WATER_IS_DARK:
    otsu_mask[roi] = (GRAY_clahe[roi] < thr).astype(np.uint8) * 255
else:
    otsu_mask[roi] = (GRAY_clahe[roi] > thr).astype(np.uint8) * 255

# fuera del polígono SIEMPRE fondo
otsu_mask[~roi] = 0

roi = ~out_img[0].mask
plt.imshow(roi, cmap="gray")
plt.title("ROI del polígono (blanco=dentro, negro=fuera)")
plt.show()

plt.figure(figsize=(6,6))
plt.imshow(otsu_mask, cmap="gray")
plt.title("Máscara Otsu (solo dentro del polígono)")
plt.axis("off")
plt.show()

# ===================================================
# 7) LIMPIEZA MORFOLÓGICA (solo dentro del ROI)
# ===================================================
water_clean = (otsu_mask == 255) & roi

water_clean = morphology.remove_small_objects(water_clean, min_size=MIN_OBJ_SIZE)
water_clean = morphology.remove_small_holes(water_clean, area_threshold=HOLE_AREA)
water_clean = morphology.binary_closing(water_clean, morphology.disk(CLOSING_DISK))

# Asegurar fuera del polígono = False
water_clean[~roi] = False

plt.figure(figsize=(7,7))
plt.imshow(water_clean, cmap="Blues")
plt.title("Máscara final (ROI + Otsu + morfología)")
plt.axis("off")
plt.show()
